In [ ]:
# Install playwright, its system dependencies, and the chromium browser
!pip install playwright
!playwright install-deps chromium
!playwright install chromium

In [2]:
import asyncio
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

async def extract_compound_data(url):
    async with async_playwright() as p:
        # Launch browser
        browser = await p.chromium.launch(headless=True)
        # Set a standard desktop viewport
        page = await browser.new_page(viewport={'width': 1280, 'height': 800})

        # Filter console logs to only show our custom progress updates
        page.on("console", lambda msg: print(f"Browser: {msg.text}") if msg.type == "log" else None)

        print(f"Navigating to: {url}")
        # Wait until network is idle to ensure initial items are loaded
        await page.goto(url, timeout=60000, wait_until="networkidle")

        container_selector = ".cards-container"
        try:
            await page.wait_for_selector(container_selector, timeout=20000)
        except Exception:
            print(f"Could not find {container_selector}. Stopping.")
            await browser.close()
            return []

        print(f"Scrolling and monitoring progress...")

        # This function handles the scrolling and returns the final unique list of links
        unique_links = await page.evaluate("""
            async () => {
                const getLinks = () => Array.from(document.querySelectorAll('a[href^="/compound/"]'))
                                            .map(a => a.href);

                let seenLinks = new Set(getLinks());
                console.log(`Initial batch: Found ${seenLinks.size} compounds.`);

                let staleCycles = 0;
                const maxStaleCycles = 5;

                while (staleCycles < maxStaleCycles) {
                    // 1. Scroll to the bottom of the page
                    window.scrollTo(0, document.body.scrollHeight);

                    // 2. Shake the specific container parent to trigger lazy-load observers
                    const container = document.querySelector('.cards-container');
                    if (container && container.parentElement) {
                        container.parentElement.scrollBy(0, 1000);
                    }

                    // 3. Wait for the lazy-load trigger and network response
                    await new Promise(resolve => setTimeout(resolve, 3500));

                    const currentLinks = getLinks();
                    let beforeCount = seenLinks.size;

                    currentLinks.forEach(link => seenLinks.add(link));

                    if (seenLinks.size > beforeCount) {
                        console.log(`Progress Update: Found ${seenLinks.size - beforeCount} more. Total unique: ${seenLinks.size}`);
                        staleCycles = 0;
                    } else {
                        // Nudge the scroll position to force observer triggers
                        window.scrollBy(0, -300);
                        await new Promise(resolve => setTimeout(resolve, 500));
                        window.scrollTo(0, document.body.scrollHeight);
                        staleCycles++;
                    }
                }
                console.log(`Scrolling finished.`);
                // Convert set back to array for Python
                return Array.from(seenLinks);
            }
        """)

        await browser.close()
        return unique_links

async def main():
    target_url = "https://www.nawy.com/search?category=compound"
    links = await extract_compound_data(target_url)

    if links:
        output_file = 'compound_links.txt'
        # Sort links for better readability in the text file
        links.sort()

        with open(output_file, 'w', encoding='utf-8') as f:
            for link in links:
                f.write(f"{link}\n")

        print(f"\nSuccess! Total unique compounds: {len(links)}")
        print(f"Full list of links saved to: {output_file}")
    else:
        print("No links were extracted.")

if __name__ == "__main__":
    try:
        asyncio.run(main())
    except RuntimeError:
        # Fallback for Jupyter/Colab
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.get_event_loop().run_until_complete(main())

Navigating to: https://www.nawy.com/search?category=compound
Scrolling and monitoring progress...
Browser: Initial batch: Found 32 compounds.
Browser: Progress Update: Found 12 more. Total unique: 44
Browser: Progress Update: Found 12 more. Total unique: 56
Browser: Progress Update: Found 12 more. Total unique: 68
Browser: Progress Update: Found 12 more. Total unique: 80
Browser: Progress Update: Found 12 more. Total unique: 92
Browser: Progress Update: Found 12 more. Total unique: 104
Browser: Progress Update: Found 12 more. Total unique: 116
Browser: Progress Update: Found 12 more. Total unique: 128
Browser: Progress Update: Found 12 more. Total unique: 140
Browser: Progress Update: Found 12 more. Total unique: 152
Browser: Progress Update: Found 12 more. Total unique: 164
Browser: Progress Update: Found 12 more. Total unique: 176
Browser: Progress Update: Found 12 more. Total unique: 188
Browser: Progress Update: Found 12 more. Total unique: 200
Browser: Progress Update: Found 12 mo

/tmp/ipykernel_636/3012888435.py:104: RuntimeWarning: coroutine 'main' was never awaited
  asyncio.get_event_loop().run_until_complete(main())
